In [39]:
from embedder import Embedder

embed = Embedder()

In [40]:
query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)
print(v.shape)
v[0]

(384,)


np.float64(-0.020582036807885073)

In [41]:
sqlitesearch_reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "02-vector-search/lessons/07-sqlitesearch-vector" in path,

)
sqlitesearch_document = [file.parse() for file in sqlitesearch_reader.read()]
sqlitesearch_content = sqlitesearch_document[0]["content"]


In [15]:

sqlitesearch_embeddings = embed.encode(sqlitesearch_content)
cosine_sim = v.dot(sqlitesearch_embeddings)
cosine_sim


np.float64(0.361070280302606)

In [42]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [45]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [46]:
print(len(chunks), len(documents))

295 72


In [ ]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_texts = [chunk["content"] for chunk in batch]
    batch_vectors = embed.encode_batch(batch_texts)
    vectors.extend(batch_vectors)



  0%|          | 0/6 [00:00<?, ?it/s]

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [58]:
print(type(vectors[0]))

<class 'numpy.ndarray'>


In [28]:
import numpy as np
X = np.array(vectors)

In [56]:
print(type(X))
X.shape

<class 'numpy.ndarray'>


(295, 384)

In [31]:
scores = X.dot(v)

In [60]:
print(type(scores))

<class 'numpy.ndarray'>


In [38]:
idx = np.argmax(scores)
print(idx)
print(scores[idx])
print(chunks[idx]["filename"])

94
0.648901732433228
02-vector-search/lessons/07-sqlitesearch-vector.md


In [71]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, chunks)

In [72]:
v_query=embed.encode("What metric do we use to evaluate a search engine?")

vindex.search(v_query, num_results=5)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['course']
)

index.fit(chunks)

question = "How do I store vectors in PostgreSQL?"

text_search_results = index.search(question, num_results=5)



In [ ]:
v_query=embed.encode("How do I store vectors in PostgreSQL?")

vector_search_results = vindex.search(v_query, num_results=5)

In [85]:
text_search_documents_list = []
vector_search_documents_list = []

In [86]:
for result in text_search_results:
    print(result["filename"])
    text_search_documents_list.append(result)
    print("-"*100)


02-vector-search/lessons/02-embeddings.md
----------------------------------------------------------------------------------------------------
03-orchestration/lessons/05-rag.md
----------------------------------------------------------------------------------------------------
02-vector-search/lessons/01-intro.md
----------------------------------------------------------------------------------------------------
03-orchestration/lessons/05-rag.md
----------------------------------------------------------------------------------------------------
02-vector-search/lessons/01-intro.md
----------------------------------------------------------------------------------------------------


In [87]:
for result in vector_search_results:
    print(result["filename"])
    vector_search_documents_list.append(result)
    print("-"*100)

02-vector-search/lessons/08-pgvector.md
----------------------------------------------------------------------------------------------------
02-vector-search/lessons/08-pgvector.md
----------------------------------------------------------------------------------------------------
03-orchestration/lessons/05-rag.md
----------------------------------------------------------------------------------------------------
02-vector-search/lessons/08-pgvector.md
----------------------------------------------------------------------------------------------------
02-vector-search/lessons/08-pgvector.md
----------------------------------------------------------------------------------------------------


In [84]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [88]:
rrf([vector_search_documents_list, text_search_documents_list])

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [89]:
rrf_query = "How do I give the model access to tools?"

rrf_encoded_query=embed.encode(rrf_query)

rrf_vector_search_results = vindex.search(rrf_encoded_query, num_results=5)

rrf_text_search_results = index.search(rrf_query, num_results=5)

rrf([rrf_vector_search_results, rrf_text_search_results])


[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 